# OpenAI Agents SDK — minimal API, three primitives

OpenAI's SDK is intentionally small. The three primitives that earn their keep:

| Primitive | What it gives you |
|---|---|
| **Handoffs** | Declarative "this agent transfers control to that one when X". |
| **Guardrails** | Input/output validators that short-circuit unsafe or off-policy runs. |
| **Sessions** | Built-in conversation memory you don't have to wire up. |

## Canonical code (with `openai-agents` installed)

```python
from agents import Agent, Runner, function_tool
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
from task import search_corpus, fetch_paper, cite

@function_tool
def search(query: str) -> list[dict]:
    return search_corpus(query, k=2)

@function_tool
def cite_claim(arxiv_id: str, claim: str) -> dict:
    return cite(arxiv_id, claim)

summariser = Agent(
    name='Summariser',
    instructions='Write a 2-sentence cited answer. Use cite_claim to verify.',
    tools=[cite_claim],
)
researcher = Agent(
    name='Researcher',
    instructions='Find the relevant paper. Hand off to Summariser when done.',
    tools=[search],
    handoffs=[summariser],
)

result = await Runner.run(researcher, input='What is RA-MoE?')
print(result.final_output)
```

In [ ]:
import os, sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'shared').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / '03-agentic-frameworks'))
if not (os.getenv('OPENAI_API_KEY') or os.getenv('ANTHROPIC_API_KEY')):
    os.environ.setdefault('LLM_CACHE_ONLY', '1')
from task import get_question
from eval import openai_agents_solve
trace = openai_agents_solve(get_question('q01'))
for step in trace.steps:
    print(f"{step.role}: {step.name or ''}  {step.output_summary or step.content or ''}"[:140])

## Tradeoffs

* **+ Small API surface** — one class, three primitives.
* **+ Tracing built in** — every run shows up in OpenAI's traces dashboard.
* **− OpenAI-flavored** — works with any OpenAI-compatible API, but the docs and examples lean OpenAI.
* **− No state graphs** — for branchy workflows, LangGraph is a better fit.